# Incident Download (Planet + GEE + SAR + DEM)

Downloads missing incident rasters using Planet orders and GEE fallback rules.
Uploads into `raw_images/raw_incidents/incident_{ID}/` in Hugging Face.

In [ ]:
import os
import re
import glob
import shutil
from datetime import datetime, timezone
from concurrent.futures import ThreadPoolExecutor, as_completed

import pandas as pd
import requests
import rasterio
from rasterio.merge import merge as rio_merge
from rasterio.warp import transform_bounds
from requests.adapters import HTTPAdapter, Retry
from kaggle_secrets import UserSecretsClient
from huggingface_hub import HfApi, CommitOperationAdd, hf_hub_download

# ----------------------------
# User configuration
# ----------------------------
INPUT_CSV = '/kaggle/input/datasets/sanjayashrestha123/landslide-reproted/landslides_from_2018_to_2026.csv'
START_IDX = 0
END_IDX = 0

PRE_DAYS = 180        # configurable, default 6 months
POST_DAYS = 30
CLOUD_MAX_AOI = 5
MAX_AOI_DEG = 0.1

ORDERS_URL = 'https://api.planet.com/compute/ops/orders/v2'
WANTED_STATES = {'success', 'partial'}
REQUEST_TIMEOUT = 300
# Each Planet order's results include multiple asset types per scene (per-scene metadata.json,
# udm2 usable-data-mask clip, AnalyticMS metadata xml, plus an order-level manifest.json) - only
# the analytic surface-reflectance GeoTIFF is the actual imagery we want to download/mosaic.
ANALYTIC_SR_SUFFIX = '_3b_analyticms_sr_clip.tif'
# UDM2 (usable data mask v2) ships in the same 'analytic_sr_udm2' bundle we order but was
# previously never downloaded/applied - band 1 of this asset is Planet's own per-pixel
# 'clear' classification (0 = cloud/shadow/haze/snow). Planet's scene-level cloud_cover
# filter (<5%) is computed over the WHOLE scene footprint, not our small clipped AOI, so a
# '5%-cloud' scene can still have its cloud concentrated right over the incident chip -
# leaving it unmasked produced washed-out white/pink blob artifacts and before/after color
# mismatches in exported chips.
UDM2_SUFFIX = '_3b_udm2_clip.tif'

# --- Monthly Mosaic Composites (alternative Planet source) ---
# True: fetch before/after imagery from Planet's monthly Mosaics/quads API instead of
# per-scene Orders API search+order+mosaic - no order submission/polling needed, and
# Planet's own compositing already removes cloud/shadow and normalizes color across
# scenes. planet_order_creation.ipynb becomes a no-op when this is True.
USE_MOSAIC_COMPOSITES = True
MOSAICS_URL = 'https://api.planet.com/basemaps/v1/mosaics'
# Must match your Planet subscription's actual monthly mosaic series name - verify with
# planet.get(MOSAICS_URL, params={'name__contains': 'monthly'}).json()['mosaics'] before
# relying on this in production.
MOSAIC_NAME_TEMPLATE = 'global_monthly_{year}_{month:02d}_mosaic'
MOSAIC_FORWARD_MONTHS = 3   # after: search incident's month then forward for first published mosaic
MOSAIC_BACKWARD_START = 1   # before: start 1 full month before the incident date
MOSAIC_BACKWARD_MONTHS = 6  # before: search up to this many further months back

MAX_WORKERS = 2          # incidents downloaded/processed concurrently - tune for Kaggle CPU/network limits
UPLOAD_BATCH_SIZE = 25   # number of ready .tif files to accumulate before flushing one batched HF commit

GEE_PROJECT = 'landslide-identification-nepal'
GEE_SERVICE_ACCOUNT = 'kaggle-import@landslide-identification-nepal.iam.gserviceaccount.com'
GEE_KEY_PATH = '/kaggle/input/datasets/sanjayashrestha123/gee-key/landslide-identification-nepal-cccd90850069.json'

HF_REPO_ID = 'sasudo2/landslides'
HF_REPO_TYPE = 'dataset'
HF_REVISION = 'main'
HF_RAW_ROOT = 'raw_images/raw_incidents'
HF_DOWNLOAD_LOG = 'raw_images/download_log.csv'
ORDER_LOG_PATH = 'raw_images/order_log.csv'  # written by planet_order_creation.ipynb

WORK_DIR = '/kaggle/working/raw_incidents'
os.makedirs(WORK_DIR, exist_ok=True)
GEE_SCALE_M = 10

In [ ]:
def clamp_aoi(min_lon, min_lat, max_lon, max_lat, max_deg=MAX_AOI_DEG):
    lon_span = max_lon - min_lon
    lat_span = max_lat - min_lat
    if lon_span <= max_deg and lat_span <= max_deg:
        return float(min_lon), float(min_lat), float(max_lon), float(max_lat)
    cx = (min_lon + max_lon) / 2.0
    cy = (min_lat + max_lat) / 2.0
    half = max_deg / 2.0
    return float(cx - half), float(cy - half), float(cx + half), float(cy + half)

def make_planet_session(api_key):
    s = requests.Session()
    s.auth = (api_key, '')
    retries = Retry(total=5, backoff_factor=2, status_forcelist=[429,500,502,503,504],
                    allowed_methods=frozenset(['GET']), respect_retry_after_header=True)
    s.mount('https://', HTTPAdapter(max_retries=retries))
    return s

def list_orders(session):
    orders = []
    url = ORDERS_URL
    while url:
        r = session.get(url, timeout=120)
        r.raise_for_status()
        data = r.json()
        orders.extend(data.get('orders', []))
        # Planet's Orders API returns the pagination link as _links.next (not _next);
        # using the wrong key silently truncated results to a single page.
        url = data.get('_links', {}).get('next')
    return orders

def extract_order_results(session, order):
    results = order.get('_links', {}).get('results')
    if results:
        return results
    oid = order.get('id')
    r = session.get(f'{ORDERS_URL}/{oid}', timeout=120)
    r.raise_for_status()
    return r.json().get('_links', {}).get('results', []) or []

def extract_order_asset_links(session, order, suffix):
    # Order results mix imagery with per-scene metadata.json/.xml, udm2 masks, and an
    # order-level manifest.json - keep only the files whose delivered name ends with `suffix`
    # (e.g. the analytic SR GeoTIFF) so we never try to mosaic a non-raster/wrong asset.
    links = []
    for r in extract_order_results(session, order):
        name = (r.get('name') or '').lower()
        loc = r.get('location')
        if loc and name.endswith(suffix.lower()):
            links.append(loc)
    return links

def extract_order_asset_pairs(session, order):
    # Pair each scene's analytic SR asset with its matching UDM2 cloud/shadow/snow/haze
    # mask by the common filename prefix (both assets for the same scene share everything
    # up to the suffix) - safer than zipping two independently-filtered lists positionally,
    # since a scene occasionally has one asset without the other.
    pairs = {}
    for r in extract_order_results(session, order):
        name = (r.get('name') or '').lower()
        loc = r.get('location')
        if not loc:
            continue
        if name.endswith(ANALYTIC_SR_SUFFIX.lower()):
            key = name[:-len(ANALYTIC_SR_SUFFIX)]
            pairs.setdefault(key, {})['sr'] = loc
        elif name.endswith(UDM2_SUFFIX.lower()):
            key = name[:-len(UDM2_SUFFIX)]
            pairs.setdefault(key, {})['udm2'] = loc
    return pairs


def mask_clear_pixels(sr_path, udm2_path):
    # Zero out any pixel Planet's own UDM2 classifier flags as cloud/shadow/haze/snow
    # (band 1 == 1 means clear) in place, so every downstream consumer (analysis grid,
    # exported chips) sees clean imagery without needing to know about the UDM2 product.
    with rasterio.open(udm2_path) as udm2:
        clear = udm2.read(1)
    with rasterio.open(sr_path) as src:
        data = src.read()
        meta = src.meta.copy()
    if clear.shape != data.shape[1:]:
        print(f'UDM2 mask shape {clear.shape} != SR shape {data.shape[1:]} for {sr_path} - skipping cloud mask')
        return
    data[:, clear == 0] = 0
    with rasterio.open(sr_path, 'w', **meta) as dst:
        dst.write(data)

def download_file(session, url, out_path):
    tmp = out_path + '.part'
    with session.get(url, stream=True, timeout=REQUEST_TIMEOUT) as r:
        r.raise_for_status()
        with open(tmp, 'wb') as f:
            for chunk in r.iter_content(chunk_size=(1 << 20)):
                if chunk:
                    f.write(chunk)
    os.replace(tmp, out_path)

def mosaic_geotiffs(src_paths, dst_path):
    srcs = [rasterio.open(p) for p in src_paths]
    try:
        mosaic, out_transform = rio_merge(srcs)
        out_meta = srcs[0].meta.copy()
        out_meta.update({
            'height': mosaic.shape[1],
            'width': mosaic.shape[2],
            'transform': out_transform,
            'count': mosaic.shape[0],
        })
        with rasterio.open(dst_path, 'w', **out_meta) as dst:
            dst.write(mosaic)
    finally:
        for s in srcs:
            s.close()

def find_mosaic_exact(session, year, month):
    name = MOSAIC_NAME_TEMPLATE.format(year=year, month=month)
    r = session.get(MOSAICS_URL, params={'name__is': name}, timeout=120)
    r.raise_for_status()
    mosaics = r.json().get('mosaics', [])
    return mosaics[0] if mosaics else None

def _shift_month(year, month, delta):
    total = year * 12 + (month - 1) + delta
    return total // 12, total % 12 + 1

def find_after_mosaic(session, incident_date, forward=MOSAIC_FORWARD_MONTHS):
    # the slide must already have happened, so only the incident's month or later qualify
    for fwd in range(forward + 1):
        y, m = _shift_month(incident_date.year, incident_date.month, fwd)
        mosaic = find_mosaic_exact(session, y, m)
        if mosaic:
            return mosaic
    return None

def find_before_mosaic(session, incident_date, start=MOSAIC_BACKWARD_START, back=MOSAIC_BACKWARD_MONTHS):
    for b in range(start, start + back):
        y, m = _shift_month(incident_date.year, incident_date.month, -b)
        mosaic = find_mosaic_exact(session, y, m)
        if mosaic:
            return mosaic
    return None

def get_quad_links(session, mosaic_id, min_lon, min_lat, max_lon, max_lat):
    links = []
    url = f'{MOSAICS_URL}/{mosaic_id}/quads'
    params = {'bbox': f'{min_lon},{min_lat},{max_lon},{max_lat}', '_page_size': 50}
    while url:
        r = session.get(url, params=params, timeout=120)
        r.raise_for_status()
        data = r.json()
        for item in data.get('items', []):
            dl = item.get('_links', {}).get('download')
            if dl:
                links.append(dl)
        url = data.get('_links', {}).get('_next')
        params = None
    return links

def download_mosaic_clip(session, mosaic, min_lon, min_lat, max_lon, max_lat, out_path, tmp_dir, tmp_tag):
    # Download every quad intersecting the AOI, mosaic them if there's more than one, then
    # window-clip to the exact incident bbox so the output extent matches the per-scene path.
    links = get_quad_links(session, mosaic['id'], min_lon, min_lat, max_lon, max_lat)
    if not links:
        return False
    parts = []
    for i, link in enumerate(links, 1):
        p = os.path.join(tmp_dir, f'_{tmp_tag}_quad_{i}.tif')
        download_file(session, link, p)
        parts.append(p)
    merged = parts[0] if len(parts) == 1 else os.path.join(tmp_dir, f'_{tmp_tag}_merged.tif')
    if len(parts) > 1:
        mosaic_geotiffs(parts, merged)
        for p in parts:
            if os.path.exists(p):
                os.remove(p)
    with rasterio.open(merged) as src:
        # Mosaic quads are delivered in the mosaic's native CRS (typically Web Mercator,
        # EPSG:3857), not WGS84 - windowing with raw lon/lat degrees against that transform
        # silently produced a near-zero-size (often literally 0x0) window and crashed on write.
        if src.crs and src.crs.to_epsg() != 4326:
            left, bottom, right, top = transform_bounds('EPSG:4326', src.crs, min_lon, min_lat, max_lon, max_lat)
        else:
            left, bottom, right, top = min_lon, min_lat, max_lon, max_lat
        window = src.window(left, bottom, right, top)
        window = window.round_offsets().round_lengths()
        if window.width < 1 or window.height < 1:
            raise ValueError(f'AOI does not intersect mosaic quad data (window={window})')
        data = src.read(window=window)
        transform = src.window_transform(window)
        meta = src.meta.copy()
        meta.update({'height': data.shape[1], 'width': data.shape[2], 'transform': transform})
    with rasterio.open(out_path, 'w', **meta) as dst:
        dst.write(data)
    if os.path.exists(merged):
        os.remove(merged)
    return True

def get_hf_existing_incident_files(api):
    existing = {}
    try:
        for f in api.list_repo_files(HF_REPO_ID, repo_type=HF_REPO_TYPE, revision=HF_REVISION):
            m = re.match(r'^raw_images/raw_incidents/incident_(\d+)/(.*)$', f)
            if m:
                inc_id = int(m.group(1))
                existing.setdefault(inc_id, set()).add(m.group(2))
    except Exception as e:
        print(f'Warning: could not list HF files: {e}')
    return existing

In [ ]:
# Auth
secrets = UserSecretsClient()
planet_api_key = secrets.get_secret('Lokesh_planet')
hf_token = secrets.get_secret('huggingface_token')

planet = make_planet_session(planet_api_key)
hf_api = HfApi(token=hf_token)

if USE_MOSAIC_COMPOSITES:
    # One-time self-check: E&R/education Planet plans don't always expose the same mosaic
    # series names as a full commercial subscription - print a sample so MOSAIC_NAME_TEMPLATE
    # can be corrected if 'global_monthly_...' doesn't match what this account actually has.
    try:
        sample = planet.get(MOSAICS_URL, params={'name__contains': 'monthly'}, timeout=60).json().get('mosaics', [])
        print(f"Sample monthly mosaic names on this account (verify MOSAIC_NAME_TEMPLATE matches): "
              f"{[m.get('name') for m in sample[:5]]}")
    except Exception as e:
        print(f'Warning: could not list sample mosaics ({e}) - verify MOSAIC_NAME_TEMPLATE manually')

# GEE init - prefer the service-account key JSON stored as a Kaggle secret (works even
# when the gee-key dataset isn't attached/mounted in this notebook run); fall back to the
# GEE_KEY_PATH file if no such secret is configured.
gee_available = False
try:
    import ee
    gee_key_json = None
    try:
        gee_key_json = secrets.get_secret('gee_json_key')
    except Exception:
        gee_key_json = None
    if gee_key_json:
        creds = ee.ServiceAccountCredentials(GEE_SERVICE_ACCOUNT, key_data=gee_key_json)
    else:
        creds = ee.ServiceAccountCredentials(GEE_SERVICE_ACCOUNT, GEE_KEY_PATH)
    ee.Initialize(creds, project=GEE_PROJECT)
    gee_available = True
    print('GEE initialized')
except Exception as e:
    print(f'GEE unavailable: {e}')

# Load incidents
df = pd.read_csv(INPUT_CSV)
if START_IDX == 0 and END_IDX == 0:
    df_sel = df.copy()
else:
    df_sel = df.iloc[START_IDX:END_IDX].copy()
df_sel['incident_on'] = pd.to_datetime(df_sel['incident_on'], dayfirst=True)
print(f'Selected incidents: {len(df_sel)}')

existing_files = get_hf_existing_incident_files(hf_api)

if USE_MOSAIC_COMPOSITES:
    # Composites don't need a submitted/polled order - process_incident() fetches
    # before/after imagery directly from the Mosaics API per incident.
    print('USE_MOSAIC_COMPOSITES=True: skipping order_log lookup - Planet before/after '
          'imagery is fetched directly per-incident from the Mosaics API in process_incident().')
    orders_by_name = {}
    all_orders_by_name = {}
else:
    # Drive incident selection + order lookup directly from order_log.csv (written by
    # planet_order_creation.ipynb) instead of scanning/regex-matching every order on the
    # account - this ties downloading to what was actually ordered.
    try:
        order_log_local = hf_hub_download(
            repo_id=HF_REPO_ID, repo_type=HF_REPO_TYPE, revision=HF_REVISION,
            filename=ORDER_LOG_PATH, token=hf_token,
        )
        order_log = pd.read_csv(order_log_local)
        print(f'Loaded order log rows: {len(order_log)}')
    except Exception as e:
        order_log = pd.DataFrame()
        print(f'Warning: could not load order log ({e}); no incidents to download')

    # Keep only the most recent attempt per (incident, order_type) that actually produced a
    # live order_id - 'failed'/'skipped_no_after'/'skipped_existing' rows never have one.
    ORDER_LOG_SKIP_STATES = {'failed', 'skipped_no_after', 'skipped_existing'}
    order_ids_from_log = {}
    if not order_log.empty:
        valid = order_log[(~order_log['order_state'].isin(ORDER_LOG_SKIP_STATES)) & (order_log['order_id'].astype(str) != '')]
        for _, r in valid.iterrows():
            key = (int(r['incident_id']), str(r['order_type']))
            order_ids_from_log[key] = str(r['order_id'])  # later rows overwrite earlier ones (most recent wins)

    incident_ids_with_orders = {inc_id for (inc_id, _) in order_ids_from_log}
    df_sel = df_sel[df_sel['id'].astype(int).isin(incident_ids_with_orders)].copy()
    print(f'Incidents with an order recorded in order_log: {len(df_sel)}')

    # Fetch current state for all orders in one bulk paginated listing (a handful of HTTP
    # calls) instead of one GET per order_id - with 1000+ logged orders, sequential per-ID
    # lookups are drastically slower and much more likely to hit rate limiting.
    needed_oids = set(order_ids_from_log.values())
    try:
        all_orders = list_orders(planet)
        orders_by_id = {o.get('id'): o for o in all_orders if o.get('id') in needed_oids}
        print(f'Fetched {len(all_orders)} order(s) via bulk list, matched {len(orders_by_id)} needed order(s)')
    except Exception as e:
        orders_by_id = {}
        print(f'Warning: could not list Planet orders: {e}')

    state_counts = {}
    for o in orders_by_id.values():
        s = o.get('state')
        state_counts[s] = state_counts.get(s, 0) + 1
    print(f'Order state breakdown (from log, total {len(orders_by_id)}): {state_counts}')

    # order_name -> order dict, mirroring the shape process_incident already expects.
    # all_orders_by_name (any state) is used only for diagnostics; orders_by_name (downloadable
    # states only) gates the actual download attempts.
    orders_by_name = {}
    all_orders_by_name = {}
    for (inc_id, order_type), oid in order_ids_from_log.items():
        o = orders_by_id.get(oid)
        if o is None:
            continue
        name = f'incident_{inc_id}_planet_{order_type}'
        all_orders_by_name[name] = o
        if o.get('state') in WANTED_STATES:
            orders_by_name[name] = o
    print(f'Orders in downloadable states: {len(orders_by_name)}')

In [ ]:
def gee_cloud_helpers():
    def mask_s2_clouds(image):
        scl = image.select('SCL')
        clean = (scl.eq(2).bitwiseOr(scl.eq(4)).bitwiseOr(scl.eq(5)).bitwiseOr(scl.eq(6)).bitwiseOr(scl.eq(7)).bitwiseOr(scl.eq(11)))
        return image.updateMask(clean)

    def add_aoi_cloud(img, aoi):
        scl = img.select('SCL')
        cloud = (scl.eq(3).Or(scl.eq(8)).Or(scl.eq(9)).Or(scl.eq(10)))
        stats = cloud.reduceRegion(reducer=ee.Reducer.mean(), geometry=aoi, scale=60, maxPixels=1e9)
        frac = stats.get('SCL')
        pct = ee.Algorithms.If(frac, ee.Number(frac).multiply(100), ee.Number(100))
        return img.set('aoi_cloud', pct)

    return mask_s2_clouds, add_aoi_cloud

def gee_download_url(image, aoi, scale=10):
    return image.getDownloadURL({'scale': scale, 'region': aoi, 'format': 'GeoTIFF', 'crs': 'EPSG:4326'})

def gee_download_to_path(image, aoi, path, scale=10):
    url = gee_download_url(image, aoi, scale=scale)
    r = requests.get(url, stream=True, timeout=300)
    r.raise_for_status()
    with open(path, 'wb') as f:
        for chunk in r.iter_content(chunk_size=8192):
            f.write(chunk)

mask_s2_clouds, add_aoi_cloud = gee_cloud_helpers()


In [ ]:
def process_incident(row):
    """Download Planet (after/before) + GEE/SAR/DEM assets for one incident.
    Runs inside a worker thread - must not mutate shared state (existing_files,
    orders_by_name, download_records, etc. are only read here).
    Returns (inc_id, inc_dir, tif_files) where tif_files is the list of locally
    downloaded .tif paths (empty if nothing was available to download yet)."""
    inc_id = int(row['id'])
    inc_dir = os.path.join(WORK_DIR, f'incident_{inc_id}')
    os.makedirs(inc_dir, exist_ok=True)

    have = existing_files.get(inc_id, set())
    before_name = f'incident_{inc_id}_planet_before'
    after_name = f'incident_{inc_id}_planet_after'
    after_ready = after_name in orders_by_name
    after_have = f'incident_{inc_id}_after.tif' in have

    min_lon, min_lat, max_lon, max_lat = clamp_aoi(row['min_lon'], row['min_lat'], row['max_lon'], row['max_lat'])
    incident_date = pd.to_datetime(row['incident_on'], dayfirst=True)

    # Planet AFTER: monthly mosaic composite (Planet already removes cloud/shadow and
    # normalizes color across scenes, so no per-scene UDM2 masking is needed here) or
    # per-scene order, gated by USE_MOSAIC_COMPOSITES.
    if not after_have and (USE_MOSAIC_COMPOSITES or after_ready):
        try:
            if USE_MOSAIC_COMPOSITES:
                mosaic = find_after_mosaic(planet, incident_date)
                if mosaic is None:
                    print(f'No published after-mosaic found for incident_{inc_id} within {MOSAIC_FORWARD_MONTHS} month(s) forward - skipping')
                else:
                    outp = os.path.join(inc_dir, f'incident_{inc_id}_after.tif')
                    ok = download_mosaic_clip(planet, mosaic, min_lon, min_lat, max_lon, max_lat, outp, inc_dir, 'after')
                    if ok:
                        print(f"Planet after ready for incident_{inc_id} (mosaic {mosaic.get('name')})")
                    else:
                        print(f"No quads found covering incident_{inc_id} AOI in after-mosaic {mosaic.get('name')}")
            else:
                order = orders_by_name[after_name]
                pairs = extract_order_asset_pairs(planet, order)
                parts = []
                for i, (key, p_urls) in enumerate(sorted(pairs.items()), 1):
                    if 'sr' not in p_urls:
                        continue
                    p = os.path.join(inc_dir, f'_after_part_{i}.tif')
                    download_file(planet, p_urls['sr'], p)
                    if 'udm2' in p_urls:
                        udm2_p = os.path.join(inc_dir, f'_after_part_{i}_udm2.tif')
                        download_file(planet, p_urls['udm2'], udm2_p)
                        mask_clear_pixels(p, udm2_p)
                        os.remove(udm2_p)
                    parts.append(p)
                if len(parts) == 1:
                    os.replace(parts[0], os.path.join(inc_dir, f'incident_{inc_id}_after.tif'))
                    print(f'Planet after ready for incident_{inc_id}')
                elif len(parts) > 1:
                    mosaic_geotiffs(parts, os.path.join(inc_dir, f'incident_{inc_id}_after.tif'))
                    for p in parts:
                        if os.path.exists(p):
                            os.remove(p)
                    print(f'Planet after ready for incident_{inc_id} ({len(parts)} scenes mosaicked)')
                else:
                    print(f'Planet after order for incident_{inc_id} has no analytic SR asset yet - skipping')
        except Exception as e:
            print(f'Planet after failed for incident_{inc_id}: {e}')

    # Planet BEFORE: monthly mosaic composite or per-scene order (see AFTER block above).
    before_have = f'incident_{inc_id}_planet_before.tif' in have
    if (USE_MOSAIC_COMPOSITES or before_name in orders_by_name) and not before_have:
        try:
            if USE_MOSAIC_COMPOSITES:
                mosaic = find_before_mosaic(planet, incident_date)
                if mosaic is None:
                    print(f'No published before-mosaic found for incident_{inc_id} within {MOSAIC_BACKWARD_MONTHS} month(s) back - skipping')
                else:
                    outp = os.path.join(inc_dir, f'incident_{inc_id}_planet_before.tif')
                    ok = download_mosaic_clip(planet, mosaic, min_lon, min_lat, max_lon, max_lat, outp, inc_dir, 'before')
                    if ok:
                        print(f"Planet before ready for incident_{inc_id} (mosaic {mosaic.get('name')})")
                    else:
                        print(f"No quads found covering incident_{inc_id} AOI in before-mosaic {mosaic.get('name')}")
            else:
                order = orders_by_name[before_name]
                pairs = extract_order_asset_pairs(planet, order)
                parts = []
                for i, (key, p_urls) in enumerate(sorted(pairs.items()), 1):
                    if 'sr' not in p_urls:
                        continue
                    p = os.path.join(inc_dir, f'_planet_before_part_{i}.tif')
                    download_file(planet, p_urls['sr'], p)
                    if 'udm2' in p_urls:
                        udm2_p = os.path.join(inc_dir, f'_planet_before_part_{i}_udm2.tif')
                        download_file(planet, p_urls['udm2'], udm2_p)
                        mask_clear_pixels(p, udm2_p)
                        os.remove(udm2_p)
                    parts.append(p)
                outp = os.path.join(inc_dir, f'incident_{inc_id}_planet_before.tif')
                if len(parts) == 1:
                    os.replace(parts[0], outp)
                    print(f'Planet before ready for incident_{inc_id}')
                elif len(parts) > 1:
                    mosaic_geotiffs(parts, outp)
                    for p in parts:
                        if os.path.exists(p):
                            os.remove(p)
                    print(f'Planet before ready for incident_{inc_id} ({len(parts)} scenes mosaicked)')
                else:
                    all_names = [r.get('name') for r in extract_order_results(planet, order)]
                    print(f'Planet before order for incident_{inc_id} has no asset matching suffix {ANALYTIC_SR_SUFFIX!r} - skipping. Result names: {all_names}')
        except Exception as e:
            print(f'Planet before failed for incident_{inc_id}: {e}')
    elif not before_have and not USE_MOSAIC_COMPOSITES:
        raw_order = all_orders_by_name.get(before_name)
        if raw_order is None:
            print(f'No Planet before order found for incident_{inc_id}')
        else:
            print(f"Planet before order for incident_{inc_id} not yet downloadable (state={raw_order.get('state')})")

    # GEE: S2 optical before fallback, slope, aspect, and SAR.
    if gee_available:
        aoi = ee.Geometry.Rectangle([min_lon, min_lat, max_lon, max_lat])
        before_start = (incident_date - pd.DateOffset(days=PRE_DAYS)).strftime('%Y-%m-%d')
        before_end = (incident_date - pd.DateOffset(days=1)).strftime('%Y-%m-%d')
        after_start = (incident_date + pd.DateOffset(days=1)).strftime('%Y-%m-%d')
        after_end = (incident_date + pd.DateOffset(days=POST_DAYS)).strftime('%Y-%m-%d')

        gee_before_have = f'incident_{inc_id}_gee_before.tif' in have
        if not gee_before_have:
            try:
                s2 = (ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
                      .filterBounds(aoi)
                      .filterDate(before_start, before_end)
                      .map(lambda img: add_aoi_cloud(img, aoi)))
                s2_ok = s2.filter(ee.Filter.lte('aoi_cloud', CLOUD_MAX_AOI))
                if s2_ok.size().getInfo() > 0:
                    best = s2_ok.sort('system:time_start', False).first()
                    gee_before_path = os.path.join(inc_dir, f'incident_{inc_id}_gee_before.tif')
                    # Select BGRN to match Planet analytic_sr band order (1=Blue,2=Green,3=Red,4=NIR)
                    gee_download_to_path(
                        mask_s2_clouds(best).select(['B2', 'B3', 'B4', 'B8']).clip(aoi),
                        aoi, gee_before_path, scale=GEE_SCALE_M)
                    print(f'GEE before ready for incident_{inc_id}')
                else:
                    print(f'No cloud-free S2 before scenes for incident_{inc_id}')
            except Exception as e:
                print(f'GEE before failed for incident_{inc_id}: {e}')

        # Each asset below is fetched in its own try/except.
        dem = ee.Image('USGS/SRTMGL1_003')
        if f'incident_{inc_id}_slope.tif' not in have:
            try:
                gee_download_to_path(ee.Terrain.slope(dem).clip(aoi), aoi, os.path.join(inc_dir, f'incident_{inc_id}_slope.tif'), scale=30)
            except Exception as e:
                print(f'GEE slope failed for incident_{inc_id}: {e}')
        if f'incident_{inc_id}_gee_aspect.tif' not in have:
            try:
                gee_download_to_path(ee.Terrain.aspect(dem).clip(aoi), aoi, os.path.join(inc_dir, f'incident_{inc_id}_gee_aspect.tif'), scale=30)
            except Exception as e:
                print(f'GEE aspect failed for incident_{inc_id}: {e}')

        try:
            s1 = (ee.ImageCollection('COPERNICUS/S1_GRD')
                  .filterBounds(aoi)
                  .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VV'))
                  .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VH'))
                  .filter(ee.Filter.eq('instrumentMode', 'IW')))
            pre_coll = s1.filterDate(before_start, before_end)
            post_coll = s1.filterDate(after_start, after_end)
            if f'incident_{inc_id}_sar_pre.tif' not in have and pre_coll.size().getInfo() > 0:
                pre_img = pre_coll.sort('system:time_start', False).first()
                gee_download_to_path(ee.Image(pre_img).select(['VV','VH']).clip(aoi), aoi, os.path.join(inc_dir, f'incident_{inc_id}_sar_pre.tif'), scale=10)
            if f'incident_{inc_id}_sar_post.tif' not in have and post_coll.size().getInfo() > 0:
                post_img = post_coll.sort('system:time_start', True).first()
                gee_download_to_path(ee.Image(post_img).select(['VV','VH']).clip(aoi), aoi, os.path.join(inc_dir, f'incident_{inc_id}_sar_post.tif'), scale=10)
        except Exception as e:
            print(f'GEE SAR failed for incident_{inc_id}: {e}')

    tif_files = sorted(glob.glob(os.path.join(inc_dir, '*.tif')))
    return inc_id, inc_dir, tif_files


download_records = []
pending_ops = []    # list[CommitOperationAdd] waiting for the next batched HF commit
pending_meta = []   # list[(inc_id, n_files, inc_dir)] describing what pending_ops holds


def flush_pending():
    """Commit every currently-queued file in a single batched HF commit, then clean up
    the local incident folders that were just uploaded. Runs only in the main thread."""
    global pending_ops, pending_meta
    if not pending_ops:
        return
    n_incidents = len(pending_meta)
    try:
        hf_api.create_commit(
            repo_id=HF_REPO_ID, repo_type=HF_REPO_TYPE, revision=HF_REVISION,
            operations=pending_ops,
            commit_message=f'Add raw images for {n_incidents} incident(s)',
        )
        for inc_id, n_files, _ in pending_meta:
            print(f'Uploaded incident_{inc_id}: {n_files} file(s) to HF (batch flush of {len(pending_ops)} files / {n_incidents} incidents)')
            download_records.append({'incident_id': inc_id, 'status': 'uploaded', 'n_files': n_files,
                                      'timestamp': datetime.now(timezone.utc).isoformat(), 'error': ''})
    except Exception as e:
        for inc_id, n_files, _ in pending_meta:
            print(f'Upload failed for incident_{inc_id}: {e}')
            download_records.append({'incident_id': inc_id, 'status': 'upload_failed', 'n_files': n_files,
                                      'timestamp': datetime.now(timezone.utc).isoformat(), 'error': str(e)})
    finally:
        for _, _, inc_dir in pending_meta:
            shutil.rmtree(inc_dir, ignore_errors=True)
        pending_ops = []
        pending_meta = []


# Skip incidents that already have every mandatory file (and no pending planet_before) up
# front, before even submitting them to the thread pool.
rows_to_process = []
for _, row in df_sel.iterrows():
    inc_id = int(row['id'])
    need = {
        f'incident_{inc_id}_after.tif',
        f'incident_{inc_id}_sar_pre.tif',
        f'incident_{inc_id}_sar_post.tif',
        f'incident_{inc_id}_slope.tif',
        f'incident_{inc_id}_gee_aspect.tif',
    }
    have = existing_files.get(inc_id, set())
    before_name = f'incident_{inc_id}_planet_before'
    # A Planet 'before' order may finish after the mandatory set is already uploaded;
    # don't permanently skip an incident that still has a pending planet_before to pick up.
    planet_before_pending = (before_name in orders_by_name) and (f'incident_{inc_id}_planet_before.tif' not in have)
    # 'before' imagery (either variant) is itself mandatory - without it 'need' being
    # satisfied does NOT mean the incident is done, since GEE-before is a cheap,
    # unconditional retry inside process_incident (no pending-order state to track).
    before_have = (f'incident_{inc_id}_planet_before.tif' in have) or (f'incident_{inc_id}_gee_before.tif' in have)
    if need.issubset(have) and before_have and not planet_before_pending:
        print(f'Skip incident_{inc_id}: all mandatory files already on HF')
        continue
    rows_to_process.append(row)

print(f'Processing {len(rows_to_process)} incident(s) with up to {MAX_WORKERS} worker(s), flushing uploads every {UPLOAD_BATCH_SIZE} file(s)')

with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
    futures = {executor.submit(process_incident, row): int(row['id']) for row in rows_to_process}
    for future in as_completed(futures):
        inc_id = futures[future]
        try:
            _, inc_dir, tif_files = future.result()
        except Exception as e:
            print(f'Incident_{inc_id} processing failed: {e}')
            download_records.append({'incident_id': inc_id, 'status': 'processing_failed', 'n_files': 0,
                                      'timestamp': datetime.now(timezone.utc).isoformat(), 'error': str(e)})
            continue

        if not tif_files:
            print(f'No files downloaded for incident_{inc_id} (no ready Planet order and/or no GEE data available yet) - skipping upload')
            download_records.append({'incident_id': inc_id, 'status': 'no_data', 'n_files': 0,
                                      'timestamp': datetime.now(timezone.utc).isoformat(), 'error': 'no source files downloaded'})
            shutil.rmtree(inc_dir, ignore_errors=True)
            continue

        for fp in tif_files:
            pending_ops.append(CommitOperationAdd(path_in_repo=f'{HF_RAW_ROOT}/incident_{inc_id}/{os.path.basename(fp)}', path_or_fileobj=fp))
        pending_meta.append((inc_id, len(tif_files), inc_dir))

        if len(pending_ops) >= UPLOAD_BATCH_SIZE:
            flush_pending()

# Final flush for any incidents left under the batch threshold
flush_pending()

# Upload download log
if len(download_records) > 0:
    dl = pd.DataFrame(download_records)
    local_dl = '/kaggle/working/download_log.csv'
    dl.to_csv(local_dl, index=False)
    hf_api.upload_file(path_or_fileobj=local_dl, path_in_repo=HF_DOWNLOAD_LOG, repo_id=HF_REPO_ID, repo_type=HF_REPO_TYPE, revision=HF_REVISION)

print('Download pass complete')
